In [56]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os

from future_land_use.util import Pipeline
from future_land_use.util.clean_unique_ids import clean_id

p = Pipeline('configs')

In [59]:
def load_overlay_layers(gdb_path, layers):
    gdf_out = gpd.GeoDataFrame()
    for layer in layers:
        # grab each layer from the geodatabase and create a juris_zn column if it doesn't exist
        gdf = gpd.read_file(gdb_path, layer=layer['layer'])
        gdf = gdf.to_crs(epsg=2285)
        if 'juris_zn_col' not in layer:
            gdf['juris_zn'] = layer['juris'] + '_' +  gdf[layer['zone_col']]
        elif 'zone_col' not in layer:
            gdf = gdf.rename(columns={layer['juris_zn_col']: 'juris_zn'})
        else:
            raise ValueError(f"Layer {layer['layer']} must have either 'juris_zn_col' or 'zone_col' specified in settings.yaml")
        gdf['juris'] = layer['juris']
        # remove any non-alphanumeric characters from the juris_zn column
        gdf['juris_zn'] = gdf['juris_zn'].apply(clean_id)
        gdf = gdf.dissolve(by='juris_zn', as_index=False)
        gdf_out = pd.concat([gdf_out, gdf[['juris','juris_zn', 'geometry']]], ignore_index=True)
    return gdf_out

global_cfg = p.settings
cfg = p.settings.get('overlay_settings', {})
input_overlay_gdb = cfg.get('overlay_gdb_path', '')

# load flu table
flu_table = p.get_table('flu_table')

# load overlay gis layers
gdf = load_overlay_layers(input_overlay_gdb, cfg.get('overlay_layers', []))

In [60]:
data_dir = p.get_data_path()

# bring in manual matches from previous runs if they exist and apply them to the gdf
manual_match_path = os.path.join(data_dir, 'overlay_manual_match.csv')
if os.path.exists(manual_match_path):
    print("loading existing manual match file")
    manual_match_df = pd.read_csv(manual_match_path)
    # save backup of the existing manual match file with a timestamp
    backup_path = os.path.join(data_dir, f"overlay_manual_match_backup_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv")
    manual_match_df.to_csv(backup_path, index=False)
    # create a mapping of juris_zn to juris_zn_manual_match and apply it to the gdf
    manual_match = manual_match_df.set_index('juris_zn')['juris_zn_manual_match']
    gdf['juris_zn_manual_match'] = gdf['juris_zn'].map(manual_match)
    # if there is a manual match, use it to update the juris_zn column in the gdf
    gdf['juris_zn'] = np.where(gdf['juris_zn_manual_match'].notnull(), gdf['juris_zn_manual_match'], gdf['juris_zn'])



loading existing manual match file


In [61]:
# merge overlay layers with flu table, flag rows that don't have a match
gdf = gdf.merge(flu_table, on='juris_zn', how='left')
gdf['match_flag'] = gdf['juris_zn'].isin(flu_table['juris_zn'])
gdf_out = gdf.loc[(gdf['match_flag']==False),'juris_zn'].to_frame()

# if the manual_match_df exists and is not empty, merge the current data to the existing manual 
# match file, otherwise create the manual_match_df from gdf_out
if 'manual_match_df' in locals() and not manual_match_df.empty:
    print("merging current data to existing manual match file")
    # remove juris_zn values that were previously unmatched but have now been fixed in the flu table
    fixed_in_table = gdf.loc[(gdf['match_flag']==True) & (gdf['juris_zn_manual_match'].isna()),'juris_zn']
    manual_match_out = manual_match_df.loc[~manual_match_df['juris_zn'].isin(fixed_in_table)].copy()
    manual_match_out = manual_match_out.merge(gdf_out, on='juris_zn', how='left')
else:
    print("creating manual match file")
    manual_match_out = gdf_out.copy()
    manual_match_out['juris_zn_manual_match'] = np.nan

# save the manual match file for review and editing
print(f"saving manual match file to {data_dir}/overlay_manual_match.csv. \n"
    "Review this file and fill in the juris_zn_manual_match column with the correct matches, \n"
    "then re-run this script to apply the manual matches to the gdf.")
manual_match_out.to_csv(f'{data_dir}/overlay_manual_match.csv', index=False)

merging current data to existing manual match file
saving manual match file to C:\Users\JKolberg\PythonProjects\PSRC\future_land_use\projects\summer_2026\data/overlay_manual_match.csv. 
Review this file and fill in the juris_zn_manual_match column with the correct matches, 
then re-run this script to apply the manual matches to the gdf.


In [62]:
gdf

,juris_x,juris_zn,geometry,juris_zn_manual_match,juris_y,Zone,definition,Bonus_avail,Bonus_included,Res_Use,...,LC_Indust,LC_Mixed,rural,ADU notes,Unnamed: 0,Unnamed: 47,Unnamed: 48,Unnamed: 49,Notes,match_flag
0,University_Place,University_Place_CCPO,"MULTIPOLYGON (((1213928.838 76290.96, 1214048....",NaN,University_Place,CCPO,An overlay for a large master-planned area own...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1,University_Place,University_Place_DIO,"POLYGON ((1210883.218 91406.079, 1210816.701 9...",NaN,University_Place,DIO,An overlay zone intended to preserve the uniqu...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
2,University_Place,University_Place_DISSO,"MULTIPOLYGON (((1208000.331 83967.081, 1207834...",NaN,University_Place,DISSO,An overlay zone intended to preserve the uniqu...,NaN,NaN,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,Poulsbo,Poulsbo_MPO,"MULTIPOLYGON (((1194885.872 276373.205, 119496...",NaN,Poulsbo,MPO,The purposes of the master plan overlay (MPO) ...,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,Kitsap,Kitsap_Silverdale_Bucklin_Hill,"POLYGON ((1181501.76 241919.819, 1181520.746 2...",Kitsap_Silverdale_Bucklin_Hill,Kitsap,Silverdale_Bucklin_Hill,NaN,Y,NaN,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
5,Kitsap,Kitsap_Silverdale_Bucklin_Hill,"POLYGON ((1181501.76 241919.819, 1181520.746 2...",Kitsap_Silverdale_Bucklin_Hill,Kitsap,Silverdale_Bucklin_Hill,NaN,Y,Y,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
6,Kitsap,Kitsap_Silverdale_Clear_Creek,"POLYGON ((1182373.757 243520.083, 1182850.384 ...",Kitsap_Silverdale_Clear_Creek,Kitsap,Silverdale_Clear_Creek,NaN,Y,NaN,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
7,Kitsap,Kitsap_Silverdale_Clear_Creek,"POLYGON ((1182373.757 243520.083, 1182850.384 ...",Kitsap_Silverdale_Clear_Creek,Kitsap,Silverdale_Clear_Creek,NaN,Y,Y,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
8,Kitsap,Kitsap_Silverdale_Kitsap_Mall,"POLYGON ((1180867.874 244172.032, 1180874.473 ...",Kitsap_Silverdale_Kitsap_Mall,Kitsap,Silverdale_Kitsap_Mall,NaN,Y,NaN,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
9,Kitsap,Kitsap_Silverdale_Kitsap_Mall,"POLYGON ((1180867.874 244172.032, 1180874.473 ...",Kitsap_Silverdale_Kitsap_Mall,Kitsap,Silverdale_Kitsap_Mall,NaN,Y,Y,Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


In [49]:
df = p.get_table('flu_imputed')

In [51]:
df.loc[df['juris_zn']=='Seattle_HR','MaxDU_Res']

1340    48.359856
Name: MaxDU_Res, dtype: float64

In [37]:
parcels = p.get_table('parcels_pts')[['parcel_id','gis_sqft']]
parcels['acres'] = (parcels['gis_sqft'] / 43560).astype(float)

In [38]:
parcel_flu = p.get_table('parcel_plan_type_xwalk')

In [39]:
flu = p.get_table('flu_imputed_hb_1110_with_overlays')

In [40]:
parcels_start = parcels.merge(parcel_flu, on='parcel_id', how='left').merge(flu, on=['juris_zn','plan_type_id'], how='left')
# res only
parcels_start = parcels_start.loc[parcels_start['Res_Use'] == 1].copy()

In [41]:
parcels_start.columns

Index(['parcel_id', 'gis_sqft', 'acres', 'juris_zn', 'plan_type_id',
       'FLU_master_id', 'Juris', 'Bonus_included', 'Res_Use', 'Comm_Use',
       'Office_Use', 'Indust_Use', 'Mixed_Use', 'FloorMaxDU_lot', 'MinDU_Res',
       'MinFAR_Comm', 'MinFAR_Office', 'MinFAR_Indust', 'MinFAR_Mixed',
       'MaxDU_Res', 'MaxFAR_Comm', 'MaxFAR_Office', 'MaxFAR_Indust',
       'MaxFAR_Mixed', 'MaxHt_Res', 'MaxHt_Comm', 'MaxHt_Office',
       'MaxHt_Indust', 'MaxHt_Mixed', 'LC_Res', 'LC_Comm', 'LC_Office',
       'LC_Indust', 'LC_Mixed', 'rural', 'SingleFamily_Use', 'MultiFamily_Use',
       'MinDU_lot', 'Notes', 'city_name', 'hb_1110_tier',
       'hb_transit_override', 'overlay_combo_id'],
      dtype='str')

In [42]:
parcels_start['dua_units'] = parcels_start['MaxDU_Res'] * parcels_start['acres']

In [43]:
parcels_start['units'] = np.where(parcels_start['dua_units'] < parcels_start['FloorMaxDU_lot'],parcels_start['FloorMaxDU_lot'],parcels_start['dua_units'])

In [44]:
start = parcels_start.groupby(['juris_zn','SingleFamily_Use','MultiFamily_Use','Mixed_Use'],dropna=False)['units'].sum().astype(int)

In [45]:
parcel_flu_overlays = p.get_table('parcel_plan_type_xwalk_with_overlays')

parcels_overlays= parcels.merge(parcel_flu_overlays, on='parcel_id', how='left').merge(flu, on=['juris_zn','plan_type_id'], how='left')
# res only
parcels_overlays = parcels_overlays.loc[parcels_overlays['Res_Use'] == 1].copy()
parcels_overlays['dua_units'] = parcels_overlays['MaxDU_Res'] * parcels_overlays['acres']
parcels_overlays['units'] = np.where(parcels_overlays['dua_units'] < parcels_overlays['FloorMaxDU_lot'],parcels_overlays['FloorMaxDU_lot'],parcels_overlays['dua_units'])
overlays = parcels_overlays.groupby(['juris_zn','SingleFamily_Use','MultiFamily_Use','Mixed_Use'],dropna=False)['units'].sum().astype(int)

In [46]:
parcel_flu_hb1110 = p.get_table('parcel_plan_type_xwalk_with_overlays_hb_1110')

# use hb1110 overlays
parcels_hb1110= parcels.merge(parcel_flu_hb1110, on='parcel_id', how='left').merge(flu, on=['juris_zn','plan_type_id'], how='left')
# res only
parcels_hb1110 = parcels_hb1110.loc[parcels_hb1110['Res_Use'] == 1].copy()
parcels_hb1110['dua_units'] = parcels_hb1110['MaxDU_Res'] * parcels_hb1110['acres']
parcels_hb1110['units'] = np.where(parcels_hb1110['dua_units'] < parcels_hb1110['FloorMaxDU_lot'],parcels_hb1110['FloorMaxDU_lot'],parcels_hb1110['dua_units'])
hb1110 = parcels_hb1110.groupby(['juris_zn','SingleFamily_Use','MultiFamily_Use','Mixed_Use'],dropna=False)['units'].sum().astype(int)

In [48]:
start.rename('units_start').to_frame().join(overlays.rename('units_overlays'), how='outer').join(hb1110.rename('units_hb1110'), how='outer').to_csv('output/overlays_hb_1110_checks.csv')